<a href="https://colab.research.google.com/github/naurasafina/python-google-colab-projek/blob/main/Klasifikasi_Judul_Berita_Convolutional_Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PREPROCESSING

In [ ]:
!pip install Sastrawi
import pandas as pd
import re
import numpy as np

from sklearn.preprocessing import LabelEncoder

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 5.0 MB/s eta 0:00:00


In [ ]:
df = pd.read_excel('DATA_DEEPLEARNING_INIII.xlsx')

df = df[['Column3', 'Column4']]
df.columns = ['judul', 'label']
df = df.dropna()

In [ ]:
def cleaning(text):
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def lowercasing(text):
    return text.lower()

stopwords = set(StopWordRemoverFactory().get_stop_words())

def stopword_removal(text):
    return " ".join(word for word in text.split() if word not in stopwords)

stemmer = StemmerFactory().create_stemmer()

def stemming(text):
    return stemmer.stem(text)

In [ ]:
df['cleaning']     = df['judul'].apply(cleaning)
df['lowercase']    = df['cleaning'].apply(lowercasing)
df['stopword']     = df['lowercase'].apply(stopword_removal)
df['stemming']     = df['stopword'].apply(stemming)

In [ ]:
tokenizer = Tokenizer(num_words=5000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['stemming'])

sequences = tokenizer.texts_to_sequences(df['stemming'])

MAX_LEN = 30
X = pad_sequences(
    sequences,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

In [ ]:
encoder = LabelEncoder()
y = encoder.fit_transform(df['label'])

In [ ]:
print("Shape X (input CNN):", X.shape)
print("Shape y (label):", y.shape)

df[['judul', 'cleaning', 'stopword', 'stemming']].head(5)

Shape X (input CNN): (358, 30)
Shape y (label): (358,)


,judul,cleaning,stopword,stemming
0,title,title,title,title
1,Kemnaker Awasi TKA di Meikarta,Kemnaker Awasi TKA di Meikarta,kemnaker awasi tka meikarta,kemnaker awas tka meikarta
2,BNI Digitalkan BNI Java Jazz 2020,BNI Digitalkan BNI Java Jazz,bni digitalkan bni java jazz,bni digital bni java jazz
3,"Terbang ke Australia, Edhy Prabowo Mau Genjot ...",Terbang ke Australia Edhy Prabowo Mau Genjot B...,terbang australia edhy prabowo mau genjot budi...,terbang australia edhy prabowo mau genjot budi...
4,OJK Siapkan Stimulus Ekonomi Antisipasi Dampak...,OJK Siapkan Stimulus Ekonomi Antisipasi Dampak...,ojk siapkan stimulus ekonomi antisipasi dampak...,ojk siap stimulus ekonomi antisipasi dampak co...


In [ ]:
df.to_csv(
    "/content/data_preprocessing_judul.csv",
    index=False
)

PEMBAGIAN DATA

In [ ]:
from sklearn.model_selection import StratifiedKFold
import numpy as np

print(X.shape, y.shape)

# Cek kelas dengan jumlah sampel < 2
unique_classes, counts = np.unique(y, return_counts=True)
single_member_classes = unique_classes[counts < 2]

if len(single_member_classes) > 0:
    print(f"Warning: Kelas dengan 1 sampel akan dikeluarkan: {single_member_classes}")
    mask = np.isin(y, single_member_classes, invert=True)
    X_filtered = X[mask]
    y_filtered = y[mask]
    print(f"Original samples: {len(y)}, After filtering: {len(y_filtered)}")
else:
    X_filtered = X
    y_filtered = y

# 5-Fold Cross Validation
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold = 1
for train_idx, test_idx in skf.split(X_filtered, y_filtered):
    X_train, X_test = X_filtered[train_idx], X_filtered[test_idx]
    y_train, y_test = y_filtered[train_idx], y_filtered[test_idx]

    print(f"\nFold {fold}")
    print("X_train:", X_train.shape)
    print("X_test :", X_test.shape)
    print("y_train:", y_train.shape)
    print("y_test :", y_test.shape)

    fold += 1

(358, 30) (358,)
Original samples: 358, After filtering: 357

Fold 1
X_train: (285, 30)
X_test : (72, 30)
y_train: (285,)
y_test : (72,)

Fold 2
X_train: (285, 30)
X_test : (72, 30)
y_train: (285,)
y_test : (72,)

Fold 3
X_train: (286, 30)
X_test : (71, 30)
y_train: (286,)
y_test : (71,)

Fold 4
X_train: (286, 30)
X_test : (71, 30)
y_train: (286,)
y_test : (71,)

Fold 5
X_train: (286, 30)
X_test : (71, 30)
y_train: (286,)
y_test : (71,)


PERANCANGAN MODEL CNN

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    MaxPooling1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam

In [ ]:
VOCAB_SIZE = 5000
EMBEDDING_DIM = 128
MAX_LEN = 30
NUM_CLASSES = len(set(y_train))

In [ ]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_LEN
    )
)

# Convolution Layer
model.add(
    Conv1D(
        filters=128,
        kernel_size=5,
        activation='relu'
    )
)

# Pooling
model.add(GlobalMaxPooling1D())

# Fully Connected
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(NUM_CLASSES, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Training Model
import numpy as np
from sklearn.preprocessing import LabelEncoder

combined_y = np.concatenate((y_train, y_test))
re_encoder_for_fit = LabelEncoder()
re_encoder_for_fit.fit(combined_y)

y_train_reindexed = re_encoder_for_fit.transform(y_train)
y_test_reindexed = re_encoder_for_fit.transform(y_test)

history = model.fit(
    X_train,
    y_train_reindexed,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test_reindexed),
    verbose=1
)

Epoch 1/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - accuracy: 0.3983 - loss: 1.0916 - val_accuracy: 0.3889 - val_loss: 1.0829
Epoch 2/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4427 - loss: 1.0528 - val_accuracy: 0.3889 - val_loss: 1.0650
Epoch 3/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.5368 - loss: 0.9977 - val_accuracy: 0.5694 - val_loss: 1.0313
Epoch 4/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8676 - loss: 0.8984 - val_accuracy: 0.7778 - val_loss: 0.9598
Epoch 5/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.9497 - loss: 0.7235 - val_accuracy: 0.7917 - val_loss: 0.8244
Epoch 6/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9829 - loss: 0.4434 - val_accuracy: 0.8889 - val_loss: 0.6406
Epoch 7/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.1901 - val_accuracy: 0.8750 - val_loss: 0.4873
Epoch 8/10
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 1.0000 - loss: 0.0615 - val_accuracy: 0.8889 - val_loss: 0.4093


EVALUASI MODEL

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test_reindexed)
print("Loss     :", loss)
print("Accuracy :", accuracy)

3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.9201 - loss: 0.3361
Loss     : 0.35511207580566406
Accuracy : 0.9027777910232544
